In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab / Linux environment)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# Distance Metric Learning: Large Margin Nearest Neighbor (LMNN) + RBF SVM Classification (`models/train_distant_analysis.ipynb`)

Loads the full emergency cohort from **`datasets/5v_cleandf.RData`** (~558,000 visits with valid ESI), performs **Large Margin Nearest Neighbor (LMNN)** metric learning from `metric_learn` across the 8 core arrival triage features, and trains a non-linear **RBF Kernel Support Vector Machine (RBF SVM)** on the learned 2D metric space to classify **`ESI 1 (Immediate Resuscitation)`** vs **`NOT ESI 1 (ESI 2–5)`**.

```mermaid
flowchart TD
    Raw["Raw Arrival Features X in R^8 (558,029 Visits)"] --> Imputer["SimpleImputer(strategy='median') & StandardScaler()"]
    Imputer --> RUS["RandomUnderSampler (1:1 Balanced Training Split: ~3.7k ESI 1 vs ~3.7k NOT ESI 1)"]
    RUS --> LMNN["Large Margin Nearest Neighbor (LMNN from metric_learn: R^8 -> R^2)"]
    LMNN --> Manifold["2D LMNN Metric Space [z1: LMNN Dim 1, z2: LMNN Dim 2]"]
    Manifold --> RBFSVM["RBF Kernel SVM Classifier (Non-Linear Margin in LMNN Metric Space)"]
    RBFSVM --> Eval["Holdout Test Evaluation (83k Visits) & Before/After Scatter Visualizations"]
```

### 🎯 Target Formulation
- **`Class 1: ESI 1 (Immediate Resuscitation)`** ($y=1$): Patients requiring immediate life-saving intervention ($5,271$ visits, $\sim 0.94\%$).
- **`Class 0: NOT ESI 1 (ESI 2–5)`** ($y=0$): Emergent, urgent, and non-urgent visits ($552,758$ visits, $\sim 99.06\%$).

### 🩺 8 Core Arrival Triage Features
1. `age`
2. `cc_breathingdifficulty`
3. `gender` (0 = Female, 1 = Male)
4. `triage_vital_hr` (Heart Rate)
5. `triage_vital_sbp` (Systolic Blood Pressure)
6. `triage_vital_dbp` (Diastolic Blood Pressure)
7. `triage_vital_rr` (Respiratory Rate)
8. `triage_vital_o2` (Oxygen Saturation - SpO2)

### 📐 Mathematical Foundation of LMNN Metric Learning
LMNN learns a linear transformation $L \in \mathbb{R}^{2 \times 8}$ that induces a Mahalanobis distance metric $D_L(x_i, x_j) = ||L(x_i - x_j)||^2$ optimizing two competing objectives:

$$\min_L \, (1 - \mu) \sum_{j \in N_k(i), y_j = y_i} ||L(x_i - x_j)||^2 + \mu \sum_{i, j, l} \xi_{ijl}$$

$$\text{s.t. } ||L(x_i - x_l)||^2 - ||L(x_i - x_j)||^2 \ge 1 - \xi_{ijl}, \quad \xi_{ijl} \ge 0, \quad \forall y_l \neq y_i$$

1. **Pull Term (Intra-Class Compactness)**: Pulls patients with the same resuscitation acuity closer together.
2. **Push Term (Inter-Class Margin)**: Enforces a margin of at least $1$ between patients of different classes, pushing away impostors.
3. **RBF SVM Decision Boundary**: The RBF kernel SVM ($K(z_i, z_j) = \exp(-\gamma ||z_i - z_j||^2)$) fits a non-linear decision boundary over the learned LMNN metric space.

In [ ]:
%%R
# ---------------------------------------------------------------------------
# Step 1: Load 5v_cleandf.RData (8 Core Triage Features, All Non-NA ESI Rows Kept)
# ---------------------------------------------------------------------------
suppressPackageStartupMessages({
  library(jsonlite)
  library(dplyr)
})

candidate_paths <- c(
  "../datasets/5v_cleandf.RData",
  "datasets/5v_cleandf.RData",
  "/kaggle/working/PKM_RF/datasets/5v_cleandf.RData",
  "/kaggle/input/5v-cleandf/5v_cleandf.RData",
  "/kaggle/input/disaster-triage-dataset/5v_cleandf.RData",
  "/kaggle/input/5v-raw/5v_cleandf.RData"
)

data_file <- NULL
for (p in candidate_paths) {
  if (file.exists(p)) {
    data_file <- p
    break
  }
}

if (is.null(data_file)) {
  stop("Could not find 5v_cleandf.RData in any candidate paths!")
}

cat(sprintf("Loading RData from: %s ...\n", data_file))
data_env <- new.env()
load(data_file, envir = data_env)
df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
raw_df   <- get(df_names[which.max(df_sizes)], envir = data_env)

cat(sprintf("Loaded %s: %d Total Rows, %d Total Columns\n", data_file, nrow(raw_df), ncol(raw_df)))

# Filter ONLY rows where ESI is not NA (retaining all ~558k observations)
valid_mask <- !is.na(raw_df$esi)
raw_df     <- raw_df[valid_mask, ]

gender_vec <- if ("gender" %in% names(raw_df)) ifelse(is.na(raw_df$gender), NA, ifelse(as.character(raw_df$gender) == "Male", 1, 0)) else rep(NA, nrow(raw_df))
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) raw_df$cc_breathingdifficulty else rep(NA, nrow(raw_df))

get_vec <- function(col_name) {
  if (col_name %in% names(raw_df)) {
    return(raw_df[[col_name]])
  } else {
    return(rep(NA, nrow(raw_df)))
  }
}

raw_esi_char <- as.character(raw_df$esi)

# Construct dataframe for 8 core triage features + ESI
df_master <- data.frame(
  age                     = raw_df$age,
  cc_breathingdifficulty  = cc_bd_vec,
  gender                  = gender_vec,
  triage_vital_hr         = get_vec("triage_vital_hr"),
  triage_vital_sbp        = get_vec("triage_vital_sbp"),
  triage_vital_dbp        = get_vec("triage_vital_dbp"),
  triage_vital_rr         = get_vec("triage_vital_rr"),
  triage_vital_o2         = get_vec("triage_vital_o2"),
  esi                     = as.numeric(raw_esi_char)
)

feature_cols <- c(
  "age", "cc_breathingdifficulty", "gender",
  "triage_vital_hr", "triage_vital_sbp", "triage_vital_dbp", "triage_vital_rr", "triage_vital_o2"
)

# Export matrices to Python (NAs preserved for SimpleImputer)
raw_mat_export <- as.matrix(df_master[, feature_cols])
esi_export     <- as.numeric(df_master$esi)

cat(sprintf("Exported Full Dataset to Python: %d rows, %d feature columns\n",
            nrow(raw_mat_export), ncol(raw_mat_export)))

In [ ]:
# ---------------------------------------------------------------------------
# Step 2: Retrieve Data from R, Partition & Apply Preprocessing
# ---------------------------------------------------------------------------
import os, json, pickle, warnings
from rpy2.robjects import r
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.utils.class_weight import compute_class_weight
from imblearn.under_sampling import RandomUnderSampler
from metric_learn import LMNN
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, recall_score, precision_score,
                             f1_score, fbeta_score, roc_auc_score, average_precision_score,
                             confusion_matrix, classification_report, roc_curve, precision_recall_curve, auc)
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
ROOT = '..' if os.path.basename(os.getcwd()) == 'models' else '.'

raw_mat_all = np.array(r('raw_mat_export'), dtype=np.float64)
esi_all     = np.array(r('esi_export'), dtype=np.int32)

FEATURES = [
    'age', 'cc_breathingdifficulty', 'gender',
    'triage_vital_hr', 'triage_vital_sbp', 'triage_vital_dbp',
    'triage_vital_rr', 'triage_vital_o2'
]

# Binary Target: 1 = ESI 1 (Immediate Resuscitation), 0 = NOT ESI 1 (ESI 2-5)
y_all = np.where(esi_all == 1, 1, 0)
LABELS = ['NOT ESI 1 (ESI 2-5)', 'ESI 1 (Resuscitation)']

print("=========================================================================")
print("     5v_cleandf COHORT FOR LMNN + RBF SVM PIPELINE")
print("=========================================================================")
print(f"Total Valid ESI Visits: {len(y_all):,}")
print(f"  * Class 0 [NOT ESI 1 (ESI 2-5)]: {np.sum(y_all == 0):,} ({np.mean(y_all == 0)*100:.2f}%)")
print(f"  * Class 1 [ESI 1 (Resuscitation)]: {np.sum(y_all == 1):,} ({np.mean(y_all == 1)*100:.2f}%)")
print(f"Features ({len(FEATURES)}): {FEATURES}")
print("=========================================================================\n")

# Stratified 3-way split: 70% Train, 15% Validation, 15% Holdout Test
itr, itmp = train_test_split(np.arange(len(y_all)), test_size=0.30, stratify=y_all, random_state=42)
iva, ite = train_test_split(itmp, test_size=0.50, stratify=y_all[itmp], random_state=42)

raw_tr  = raw_mat_all[itr]
raw_val = raw_mat_all[iva]
raw_te  = raw_mat_all[ite]

y_train = y_all[itr]
y_val   = y_all[iva]
y_test  = y_all[ite]

# Compute inverse class weights on Training partition
classes = np.array([0, 1])
computed_weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
class_weight_dict = {0: float(computed_weights[0]), 1: float(computed_weights[1])}

print(f"✓ Computed Inverse Class Weights on Training Set:")
print(f"  * Weight for NOT ESI 1 (Class 0): {class_weight_dict[0]:.4f}")
print(f"  * Weight for ESI 1 (Class 1)    : {class_weight_dict[1]:.4f} (Ratio: {class_weight_dict[1]/class_weight_dict[0]:.2f}x penalty)")

# Fit SimpleImputer & StandardScaler strictly on Training partition
print("\nFitting SimpleImputer(strategy='median') & StandardScaler on Training set...")
imputer = SimpleImputer(strategy='median')
X_tr_imp  = imputer.fit_transform(raw_tr)
X_val_imp = imputer.transform(raw_val)
X_te_imp  = imputer.transform(raw_te)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_tr_imp)
X_val_scaled   = scaler.transform(X_val_imp)
X_test_scaled  = scaler.transform(X_te_imp)

print(f"✓ Partition Shapes: Train={X_train_scaled.shape}, Val={X_val_scaled.shape}, Test={X_test_scaled.shape}")

In [ ]:
# ---------------------------------------------------------------------------
# Step 3: Train Large Margin Nearest Neighbor (LMNN) Metric Learning
# ---------------------------------------------------------------------------
print("Applying Balanced Subsampling for LMNN Metric Learning Optimization...")

# 1:1 Balanced sampling on training partition for clean metric learning
rus = RandomUnderSampler(sampling_strategy=1.0, random_state=42)
X_tr_res, y_tr_res = rus.fit_resample(X_train_scaled, y_train)
print(f"✓ LMNN Training Subsample: {len(y_tr_res):,} visits ({np.sum(y_tr_res == 1):,} ESI 1 vs {np.sum(y_tr_res == 0):,} NOT ESI 1)")

print("\nFitting Large Margin Nearest Neighbor (LMNN: R^8 -> R^2)...")
lmnn = LMNN(
    k=5,
    n_components=2,
    max_iter=100,
    random_state=42,
    verbose=True
)
lmnn.fit(X_tr_res, y_tr_res)

# Extract learned LMNN linear transformation matrix L (2 x 8)
L_mat = lmnn.components_
print(f"\n✓ LMNN Linear Metric Transformation Matrix L Shape: {L_mat.shape}")

# Transform all partitions to the learned 2D LMNN Metric Space
print("Projecting feature spaces through learned LMNN metric: R^8 -> R^2...")
Z_train = lmnn.transform(X_train_scaled)
Z_val   = lmnn.transform(X_val_scaled)
Z_test  = lmnn.transform(X_test_scaled)
Z_tr_res = lmnn.transform(X_tr_res)

# 2D Linear PCA of Raw 8 Features (for Before-Projection Baseline)
pca_raw = PCA(n_components=2, random_state=42)
X_raw_pca_te = pca_raw.fit_transform(X_test_scaled)

print(f"✓ 2D LMNN Metric Coordinates: Train={Z_train.shape}, Val={Z_val.shape}, Test={Z_test.shape}")

In [ ]:
# ---------------------------------------------------------------------------
# Step 4: Train RBF Kernel SVM Classifier on the 2D LMNN Metric Space
# ---------------------------------------------------------------------------
print("Training RBF Kernel SVM Classifier on 2D LMNN Metric Space...")

# Fit RBF Kernel SVM on the balanced LMNN metric representations
rbf_svm = SVC(
    kernel='rbf',
    C=1.5,
    gamma='scale',
    class_weight='balanced',
    probability=True,
    random_state=42
)
rbf_svm.fit(Z_tr_res, y_tr_res)

# Calibrate RBF SVM posterior probabilities on the unaltered validation partition
print("Calibrating RBF SVM on Validation Partition (Platt/Sigmoid Scaling)...")
svm_calibrated = CalibratedClassifierCV(rbf_svm, method='sigmoid', cv='prefit')
svm_calibrated.fit(Z_val, y_val)

print(f"\n✓ RBF SVM Classifier Successfully Trained on LMNN Metric Space!")
print(f"  * Kernel Function : Radial Basis Function (RBF)")
print(f"  * Support Vectors : {len(rbf_svm.support_)} support vectors across {len(Z_tr_res)} training samples")
print(f"  * Regularization C: 1.5 | Gamma: scale")

In [ ]:
# ---------------------------------------------------------------------------
# Step 5: Holdout Test Set Evaluation of LMNN + RBF SVM (83,705 Visits)
# ---------------------------------------------------------------------------
p_test_esi1 = svm_calibrated.predict_proba(Z_test)[:, 1]
pred_test   = (p_test_esi1 >= 0.50).astype(int)
dec_func_te = rbf_svm.decision_function(Z_test)

acc      = accuracy_score(y_test, pred_test)
bal_acc  = balanced_accuracy_score(y_test, pred_test)
rec_esi1 = recall_score(y_test == 1, pred_test == 1, zero_division=0)
spec_not = recall_score(y_test == 0, pred_test == 0, zero_division=0)
prec_esi1= precision_score(y_test == 1, pred_test == 1, zero_division=0)
f1_esi1  = f1_score(y_test == 1, pred_test == 1, zero_division=0)
f05_esi1 = fbeta_score(y_test == 1, pred_test == 1, beta=0.5, zero_division=0)
auc_val  = roc_auc_score(y_test, p_test_esi1)
pr_auc   = average_precision_score(y_test, p_test_esi1)

report_data = [{
    'Model': 'LMNN Metric Learning + RBF SVM',
    'Metric_Space': '2D LMNN Space [z1, z2]',
    'Kernel': 'RBF (Gaussian)',
    'Accuracy': round(acc, 4),
    'Balanced_Accuracy': round(bal_acc, 4),
    'ESI1_Sensitivity (Recall)': round(rec_esi1, 4),
    'Specificity (NOT ESI 1 Recall)': round(spec_not, 4),
    'ESI1_Precision': round(prec_esi1, 4),
    'ESI1_F1': round(f1_esi1, 4),
    'ESI1_F0.5': round(f05_esi1, 4),
    'ROC_AUC': round(auc_val, 4),
    'PR_AUC': round(pr_auc, 4)
}]

report_df = pd.DataFrame(report_data)
print("=====================================================================================================================")
print("         HOLDOUT TEST EVALUATION: LMNN METRIC LEARNING + RBF SVM CLASSIFIER")
print("=====================================================================================================================")
print(f"Total Test Cohort Evaluated: {len(y_test):,} visits ({np.sum(y_test == 1):,} ESI 1 vs {np.sum(y_test == 0):,} NOT ESI 1)\n")
print(report_df.to_string(index=False))
print("=====================================================================================================================\n")

print("Detailed Classification Report:")
print(classification_report(y_test, pred_test, target_names=LABELS, digits=4))

reports_dir = f'{ROOT}/reports'
os.makedirs(reports_dir, exist_ok=True)
report_file = os.path.join(reports_dir, 'lmnn_rbf_svm_report.csv')
report_df.to_csv(report_file, index=False)
print(f"Metrics report saved to: {report_file}")

In [ ]:
# ---------------------------------------------------------------------------
# Step 6: Confusion Matrix Heatmap for LMNN + RBF SVM
# ---------------------------------------------------------------------------
plots_dir = f'{ROOT}/plots'
os.makedirs(plots_dir, exist_ok=True)
os.makedirs(os.path.join(plots_dir, 'distant_analysis'), exist_ok=True)
os.makedirs(os.path.join(plots_dir, 'image'), exist_ok=True)

cm = confusion_matrix(y_test, pred_test, labels=[0, 1])
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, ax = plt.subplots(figsize=(7.5, 6))
annot = np.empty_like(cm, dtype=object)
for i in range(2):
    for j in range(2):
        annot[i, j] = f"{cm[i, j]:,}\n({cm_norm[i, j]*100:.2f}%)"

sns.heatmap(
    cm_norm,
    annot=annot,
    fmt='',
    cmap='Blues',
    cbar=True,
    ax=ax,
    vmin=0,
    vmax=1,
    xticklabels=['NOT ESI 1 (ESI 2-5)', 'ESI 1 (Resuscitation)'],
    yticklabels=['NOT ESI 1 (ESI 2-5)', 'ESI 1 (Resuscitation)']
)

ax.set_title(
    f'LMNN Metric Learning + RBF SVM: Confusion Matrix (Holdout Test)\n'
    f'Accuracy: {acc*100:.2f}% | ESI 1 Sensitivity: {rec_esi1*100:.2f}% | Specificity: {spec_not*100:.2f}%',
    fontsize=11.5,
    fontweight='bold',
    pad=12
)
ax.set_xlabel('Predicted Acuity Label', fontsize=11, fontweight='bold')
ax.set_ylabel('True Acuity Label', fontsize=11, fontweight='bold')

plt.tight_layout()
cm_plot_path = os.path.join(plots_dir, 'distant_analysis', 'lmnn_rbf_svm_confusion_matrix.png')
plt.savefig(cm_plot_path, dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(plots_dir, 'image', 'lmnn_rbf_svm_confusion_matrix.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"Confusion Matrix saved to: {cm_plot_path}")

In [ ]:
# ---------------------------------------------------------------------------
# Step 7: BEFORE vs AFTER Scatter Plots & RBF SVM Non-Linear Boundary Contours
# ---------------------------------------------------------------------------
np.random.seed(42)
test_esi1_idx = np.where(y_test == 1)[0]
test_not_idx  = np.where(y_test == 0)[0]
test_not_sample = np.random.choice(test_not_idx, min(2500, len(test_not_idx)), replace=False)

fig, axes = plt.subplots(2, 2, figsize=(17, 14))

# Panel 1: BEFORE - Heart Rate vs Systolic Blood Pressure
hr_idx  = FEATURES.index('triage_vital_hr')
sbp_idx = FEATURES.index('triage_vital_sbp')

axes[0, 0].scatter(X_te_imp[test_not_sample, hr_idx], X_te_imp[test_not_sample, sbp_idx],
                   c='#1f77b4', alpha=0.35, s=18, label='NOT ESI 1 (Sample)')
axes[0, 0].scatter(X_te_imp[test_esi1_idx, hr_idx], X_te_imp[test_esi1_idx, sbp_idx],
                   c='#d62728', alpha=0.85, s=36, edgecolors='black', linewidth=0.5, label='ESI 1 (Resuscitation)')
axes[0, 0].set_title('BEFORE: Raw Vital Features (Heart Rate vs Systolic BP)\n[Heavy Class Overlap & Non-Linear Entanglement]', fontsize=11.5, fontweight='bold', pad=10)
axes[0, 0].set_xlabel('Heart Rate (bpm)', fontsize=10.5, fontweight='bold')
axes[0, 0].set_ylabel('Systolic BP (mmHg)', fontsize=10.5, fontweight='bold')
axes[0, 0].set_xlim(30, 200)
axes[0, 0].set_ylim(50, 240)
axes[0, 0].grid(True, linestyle=':', alpha=0.4)
axes[0, 0].legend(loc='upper right', fontsize=9.5, frameon=True, framealpha=0.9)

# Panel 2: BEFORE - Respiratory Rate vs SpO2
rr_idx = FEATURES.index('triage_vital_rr')
o2_idx = FEATURES.index('triage_vital_o2')

axes[0, 1].scatter(X_te_imp[test_not_sample, rr_idx], X_te_imp[test_not_sample, o2_idx],
                   c='#1f77b4', alpha=0.35, s=18, label='NOT ESI 1 (Sample)')
axes[0, 1].scatter(X_te_imp[test_esi1_idx, rr_idx], X_te_imp[test_esi1_idx, o2_idx],
                   c='#d62728', alpha=0.85, s=36, edgecolors='black', linewidth=0.5, label='ESI 1 (Resuscitation)')
axes[0, 1].set_title('BEFORE: Raw Vital Features (Respiratory Rate vs SpO2)\n[Severe Overlap in Clinical Oxygenation Ranges]', fontsize=11.5, fontweight='bold', pad=10)
axes[0, 1].set_xlabel('Respiratory Rate (bpm)', fontsize=10.5, fontweight='bold')
axes[0, 1].set_ylabel('Oxygen Saturation SpO2 (%)', fontsize=10.5, fontweight='bold')
axes[0, 1].set_xlim(6, 45)
axes[0, 1].set_ylim(70, 100)
axes[0, 1].grid(True, linestyle=':', alpha=0.4)
axes[0, 1].legend(loc='lower left', fontsize=9.5, frameon=True, framealpha=0.9)

# Panel 3: BEFORE - 2D Linear PCA of Raw 8 Features
axes[1, 0].scatter(X_raw_pca_te[test_not_sample, 0], X_raw_pca_te[test_not_sample, 1],
                   c='#1f77b4', alpha=0.35, s=18, label='NOT ESI 1 (Sample)')
axes[1, 0].scatter(X_raw_pca_te[test_esi1_idx, 0], X_raw_pca_te[test_esi1_idx, 1],
                   c='#d62728', alpha=0.85, s=36, edgecolors='black', linewidth=0.5, label='ESI 1 (Resuscitation)')
axes[1, 0].set_title('BEFORE: 2D Linear PCA Baseline (Original 8-D Space)\n[Linear Unsupervised Projection Fails to Separate Classes]', fontsize=11.5, fontweight='bold', pad=10)
axes[1, 0].set_xlabel('Linear Principal Component 1', fontsize=10.5, fontweight='bold')
axes[1, 0].set_ylabel('Linear Principal Component 2', fontsize=10.5, fontweight='bold')
axes[1, 0].grid(True, linestyle=':', alpha=0.4)
axes[1, 0].legend(loc='upper right', fontsize=9.5, frameon=True, framealpha=0.9)

# Panel 4: AFTER - 2D LMNN Metric Space with RBF SVM Decision Boundary Contours
z1_min, z1_max = Z_test[:, 0].min() - 0.5, Z_test[:, 0].max() + 0.5
z2_min, z2_max = Z_test[:, 1].min() - 0.5, Z_test[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(z1_min, z1_max, 250), np.linspace(z2_min, z2_max, 250))
grid_z = np.c_[xx.ravel(), yy.ravel()]

# RBF SVM probability contours on grid
probs_grid = svm_calibrated.predict_proba(grid_z)[:, 1].reshape(xx.shape)
cf = axes[1, 1].contourf(xx, yy, probs_grid, levels=np.linspace(0, 1, 11), cmap='RdYlBu_r', alpha=0.60, vmin=0, vmax=1)

# RBF SVM Non-Linear Decision Boundary (Solid Black Line at P=0.50)
cs_decision = axes[1, 1].contour(xx, yy, probs_grid, levels=[0.50], colors='black', linewidths=2.8, linestyles='-')
axes[1, 1].clabel(cs_decision, inline=True, fontsize=10, fmt='RBF SVM Cutoff (P=0.50)')

# Overlay Test Scatter Points
axes[1, 1].scatter(Z_test[test_not_sample, 0], Z_test[test_not_sample, 1],
                   c='#1f77b4', alpha=0.35, s=18, label='NOT ESI 1 (Sample)')
axes[1, 1].scatter(Z_test[test_esi1_idx, 0], Z_test[test_esi1_idx, 1],
                   c='#d62728', alpha=0.85, s=40, edgecolors='black', linewidth=0.6, label='ESI 1 (Resuscitation)')

axes[1, 1].set_title('AFTER: 2D LMNN Large Margin Metric Space + RBF SVM Boundary\n[Metric Learn LMNN (k=5) + Non-Linear RBF Kernel SVM]', fontsize=11.5, fontweight='bold', pad=10)
axes[1, 1].set_xlabel('Component 1: LMNN Metric Dimension 1 (z1)', fontsize=10.5, fontweight='bold')
axes[1, 1].set_ylabel('Component 2: LMNN Metric Dimension 2 (z2)', fontsize=10.5, fontweight='bold')
axes[1, 1].set_xlim(z1_min, z1_max)
axes[1, 1].set_ylim(z2_min, z2_max)
axes[1, 1].grid(True, linestyle=':', alpha=0.4)
axes[1, 1].legend(loc='upper right', fontsize=9.5, frameon=True, framealpha=0.9)
cbar = plt.colorbar(cf, ax=axes[1, 1], fraction=0.046, pad=0.04)
cbar.set_label('RBF SVM Predicted P(ESI 1)', fontsize=9.5, fontweight='bold')

plt.suptitle('Distance Metric Learning: Large Margin Nearest Neighbor (LMNN) & RBF SVM Classification\nBEFORE vs AFTER Comparison on Holdout Test Set', fontsize=14.5, fontweight='bold', y=0.995)
plt.tight_layout()

scatter_file = os.path.join(plots_dir, 'distant_analysis', 'lmnn_rbf_svm_before_after_scatter.png')
plt.savefig(scatter_file, dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(plots_dir, 'image', 'lmnn_rbf_svm_before_after_scatter.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"Before/After LMNN RBF SVM scatter plot saved to: {scatter_file}")

In [ ]:
# ---------------------------------------------------------------------------
# Step 8: RBF SVM Decision Function Margin & ROC / PR Curves
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Panel 1: RBF SVM Decision Function Distribution
sns.kdeplot(dec_func_te[y_test == 0], ax=axes[0], color='#1f77b4', fill=True, alpha=0.35, linewidth=2.0, label='NOT ESI 1 (ESI 2-5)')
sns.kdeplot(dec_func_te[y_test == 1], ax=axes[0], color='#d62728', fill=True, alpha=0.35, linewidth=2.0, label='ESI 1 (Resuscitation)')
axes[0].axvline(x=0, color='black', linestyle='--', linewidth=2.0, label='RBF SVM Decision Boundary (Margin = 0)')
axes[0].set_title('RBF SVM Decision Function Margin on LMNN Metric Space\n[Signed Distance to Non-Linear Separating Boundary]', fontsize=12, fontweight='bold', pad=10)
axes[0].set_xlabel('Signed Decision Function Score', fontsize=10.5, fontweight='bold')
axes[0].set_ylabel('Probability Density', fontsize=10.5, fontweight='bold')
axes[0].grid(True, linestyle=':', alpha=0.4)
axes[0].legend(loc='upper right', fontsize=10)

# Panel 2: ROC Curve for LMNN + RBF SVM
fpr, tpr, _ = roc_curve(y_test, p_test_esi1)
axes[1].plot(fpr, tpr, color='#9467bd', linewidth=2.4, label=f'LMNN + RBF SVM (ROC-AUC = {auc_val:.4f})')
axes[1].plot([0, 1], [0, 1], 'k--', color='gray', linewidth=1.2, label='Random Chance (0.5000)')
axes[1].set_title('ROC Curve: LMNN Metric Learning + RBF SVM (Holdout Test)', fontsize=12, fontweight='bold', pad=10)
axes[1].set_xlabel('False Positive Rate (1 - Specificity)', fontsize=10.5, fontweight='bold')
axes[1].set_ylabel('True Positive Rate (Sensitivity)', fontsize=10.5, fontweight='bold')
axes[1].grid(True, linestyle=':', alpha=0.4)
axes[1].legend(loc='lower right', fontsize=10.5)

plt.tight_layout()
eval_file = os.path.join(plots_dir, 'distant_analysis', 'lmnn_rbf_svm_eval_curves.png')
plt.savefig(eval_file, dpi=300, bbox_inches='tight')
plt.show()
print(f"Evaluation curves saved to: {eval_file}")

In [ ]:
# ---------------------------------------------------------------------------
# Step 9: Export Production LMNN + RBF SVM Deployment Bundle & Manifest
# ---------------------------------------------------------------------------
deploy_dir = f'{ROOT}/deploy'
os.makedirs(deploy_dir, exist_ok=True)

lmnn_svm_bundle = {
    'imputer': imputer,
    'scaler': scaler,
    'lmnn': lmnn,
    'L_matrix': L_mat,
    'svm_model': svm_calibrated,
    'rbf_svm': rbf_svm,
    'features': FEATURES,
    'labels': LABELS
}

bundle_file = os.path.join(deploy_dir, 'lmnn_rbf_svm_bundle.pkl')
with open(bundle_file, 'wb') as f:
    pickle.dump(lmnn_svm_bundle, f)

manifest = dict(
    pipeline_architecture='LMNN_Distance_Metric_Learning_Plus_RBF_SVM',
    metric_method='Large_Margin_Nearest_Neighbor_LMNN',
    lmnn_k_neighbors=5,
    lmnn_output_dimensions=2,
    transformation_matrix_shape=list(L_mat.shape),
    classifier='SVC_RBF_Kernel_Balanced_Calibrated',
    rbf_kernel_C=1.5,
    rbf_gamma='scale',
    dataset='5v_cleandf_RData',
    features=FEATURES,
    target='ESI1_vs_NOT_ESI1',
    total_samples=len(y_all),
    n_esi1=int(np.sum(y_all == 1)),
    n_not_esi1=int(np.sum(y_all == 0)),
    holdout_metrics=report_data[0]
)

manifest_file = os.path.join(deploy_dir, 'lmnn_rbf_svm_manifest.json')
with open(manifest_file, 'w') as f:
    json.dump(manifest, f, indent=2)

print(f"✓ Exported LMNN + RBF SVM Bundle  : {bundle_file}")
print(f"✓ Exported LMNN + RBF SVM Manifest: {manifest_file}")